In [2]:

# Uppdaterar ändringar i src utan att behöva starta om kernel:

%load_ext autoreload 
%autoreload 2
%matplotlib inline


import sys
import os
import time

# Check kernel
print(f'path: {sys.executable}')

from typing import Tuple
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from heston_project.models.DDN import *
from heston_project.utils import *

# Absolut path:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..' ))

print(project_root)

# Lägg till i sys path
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added '{project_root}' to sys.path")

# Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)

print("Setup Complete.")


path: /Users/manswestman/Kandidatarbete/venv/bin/python
/Users/manswestman/Kandidatarbete
Added '/Users/manswestman/Kandidatarbete' to sys.path
Using device: cpu
Setup Complete.


In [9]:
project_root = Path(project_root) 
df = pd.read_csv(project_root/'data'/'full_dataset_training_200000(2).csv')
print(df.columns)
# df = pd.read_csv(project_root/'data'/'dataset_100K_nofeller.csv')

# Shuffle with a fixed random state for reproducibility
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_scaler = df.copy()

#Scale theta & v0 (Ska dessa behållas i vanlig form också för att inte scalea om i output) ÖVERVÄG KOPIERA
df_scaler[['theta', 'v0']] = np.sqrt(df_scaler[['theta', 'v0']])
sqrt_copy = df_scaler[['theta', 'v0']].copy()

inputs = df_scaler[['kappa', 'theta', 'v0', 'rho', 'sigma', 'r', 'lm', 'T']]
labels = df_scaler[['price', 'dtheta', 'dsigma', 'drho', 'dkappa', 'dv0']].copy()
print(max(np.exp(-inputs['lm'])))
# See utils for current param bounds

# param_bounds = {
#     "lm":    [-1.5, 1.5],
#     "r":     [-0.01, 0.10],
#     "T":     [1/52, 3.0],
#     "theta": [0.0, 1.0],
#     "sigma": [0.1, 1.5],
#     "rho":   [-0.95, 0.0],
#     "kappa": [0.05, 5.0],
#     "v0":    [0.0, 1.0],
# }

pmin = labels['price'].min()
pmax = labels['price'].max()

# Skalar inputs till [-1,1]
inputs = scale_inputs(inputs, zero_centered=True)

# Skalar price till [0,1]
labels['price'] = scale_price(labels['price'], zero_centered=False)


# Skalar greek targets till dP_tilde/dx (där x=inputs skalat till [-1,1]) genom kedjeregeln
def scale_greek_target(greek, xmin, xmax, p_max, p_min):
    return ((xmax - xmin) * greek) / (2 * (p_max - p_min))

greek_max_mins = {}
for k, (kmin, kmax) in param_bounds.items():
    greek_label = 'd' + str(k)
    print(param_bounds)

    if k in ["theta", "v0"] and greek_label in labels.columns: #En extra term i kedjeregeln krävs för att ta hänsyn till sqrt-transformationen
        labels[greek_label] = scale_greek_target(labels[greek_label], kmin, kmax, pmax, pmin) * (2 * sqrt_copy[k])
    elif greek_label in labels.columns:
        labels[greek_label] = scale_greek_target(labels[greek_label], kmin, kmax, pmax, pmin)
    if greek_label in labels.columns:
        greek_max_mins[greek_label] = [labels[greek_label].max()]
        greek_max_mins[greek_label].append(labels[greek_label].min())


#Split skalat data
X_train, X_val, y_train, y_val = train_test_split(inputs, labels, test_size=0.1, random_state=42)

print(max(y_train['price']), min(y_train['price']))

train_dataset = TensorDataset(
    torch.tensor(X_train.values, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32)
    )
val_dataset = TensorDataset(
    torch.tensor(X_val.values, dtype=torch.float32),
    torch.tensor(y_val.values, dtype=torch.float32)
    )

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2048)


Index(['kappa', 'theta', 'sigma', 'rho', 'v0', 'T', 'r', 'lm', 'S', 'K',
       'price', 'dtheta', 'dv0', 'dsigma', 'drho', 'dkappa', 'stable', 'IV',
       'vega', 'IVdkappa', 'IVdtheta', 'IVdsigma', 'IVdrho', 'IVdv0'],
      dtype='str')
4.481674645369087
{'lm': [-1.5, 1.5], 'r': [-0.01, 0.1], 'T': [0.019230769230769232, 3.0], 'theta': [0.0, 1.0], 'sigma': [0.1, 1.5], 'rho': [-0.95, 0.0], 'kappa': [0.05, 5.0], 'v0': [0.0, 1.0]}
{'lm': [-1.5, 1.5], 'r': [-0.01, 0.1], 'T': [0.019230769230769232, 3.0], 'theta': [0.0, 1.0], 'sigma': [0.1, 1.5], 'rho': [-0.95, 0.0], 'kappa': [0.05, 5.0], 'v0': [0.0, 1.0]}
{'lm': [-1.5, 1.5], 'r': [-0.01, 0.1], 'T': [0.019230769230769232, 3.0], 'theta': [0.0, 1.0], 'sigma': [0.1, 1.5], 'rho': [-0.95, 0.0], 'kappa': [0.05, 5.0], 'v0': [0.0, 1.0]}
{'lm': [-1.5, 1.5], 'r': [-0.01, 0.1], 'T': [0.019230769230769232, 3.0], 'theta': [0.0, 1.0], 'sigma': [0.1, 1.5], 'rho': [-0.95, 0.0], 'kappa': [0.05, 5.0], 'v0': [0.0, 1.0]}
{'lm': [-1.5, 1.5], 'r': [-0.01, 0.1],

**Weights for greek loss**

In [3]:
greek_w = {}
raw_weights = []
keys_in_order = []

# 1. Iterate through both the key and the [max, min] list correctly
for k, (max_val, min_val) in greek_max_mins.items():
    
    # Calculate range
    val_range = max_val - min_val
    
    # Calculate raw weight
    raw_weight = 1.0 / (val_range**2 + 1e-8)
    
    raw_weights.append(raw_weight)
    keys_in_order.append(k)

# 2. Convert the Python list to a PyTorch tensor BEFORE doing math
raw_weights_tensor = torch.tensor(raw_weights, dtype=torch.float32)

# 3. Normalize the tensor
normalized_weights_tensor = raw_weights_tensor / torch.mean(raw_weights_tensor)

# 4. Map them back to the dictionary for easy lookup
for i, k in enumerate(keys_in_order):
    # .item() extracts the standard Python float from the 0D tensor
    greek_w[f'{k}'] = normalized_weights_tensor[i].item() 

print(greek_w)

{'dtheta': 0.7582466006278992, 'dsigma': 0.9831393957138062, 'drho': 2.4271323680877686, 'dkappa': 0.008703718893229961, 'dv0': 0.8227781653404236}


OBS NOTERA NYA VIKTER!

In [4]:
import torch

greek_w = {}
raw_weights = []
keys_in_order = []

# Iterate through the keys in your param_bounds
for k in param_bounds.keys():
    greek_label = 'd' + str(k)
    
    # Check if this greek target was included in your dataset
    if greek_label in y_train.columns:
        
        # Calculate variance directly from the training set to avoid data leakage
        col_variance = y_train[greek_label].var()
        
        # Calculate raw weight using inverse variance
        raw_weight = 1.0 / (col_variance + 1e-8)
        
        raw_weights.append(raw_weight)
        keys_in_order.append(greek_label)

# Convert to a PyTorch tensor
raw_weights_tensor = torch.tensor(raw_weights, dtype=torch.float32)

# Normalize the weights so their mean is 1.0
normalized_weights_tensor = raw_weights_tensor / torch.mean(raw_weights_tensor)

# Map them back to the dictionary for easy lookup in your loss function
for i, key in enumerate(keys_in_order):
    greek_w[key] = normalized_weights_tensor[i].item()

print("Inverse Variance Weights (computed from y_train):\n", greek_w)

Inverse Variance Weights (computed from y_train):
 {'dtheta': 0.08659274131059647, 'dsigma': 2.0540809631347656, 'drho': 2.374021053314209, 'dkappa': 0.11836783587932587, 'dv0': 0.3669370114803314}


**Training & Validation Steps**

In [5]:
# Bestämmer vilka greeks vi plockar ut för pred_grads och y_grads som ska användas i lossfunktionen.
relevant_greeks = ['kappa', 'v0', 'rho', 'sigma', 'theta']
# y och full_grad bygger på dataset med icke-matchande kolumnordning     
input_columns = list(inputs.columns)
output_columns = list(labels.columns)

index_in_input = torch.as_tensor([input_columns.index(k) for k in relevant_greeks], device=device)
index_in_output = torch.as_tensor([output_columns.index(f'd{k}') for k in relevant_greeks], device=device)

weights_list = [greek_w[f'd{k}'] for k in relevant_greeks]
greek_weights_tensor = torch.tensor(weights_list, dtype=torch.float32, device=device)

In [6]:
def train_step(model, criterion, batch: tuple, optimizer, weights):
        
        x, y = batch

        x.requires_grad_(True) # "håll koll på gradienter"

        optimizer.zero_grad()

        # Price loss
        pred_price = model(x)
        y_price = y[:, 0:1]
        l_p = criterion(pred_price, y_price)

        # Compute gradients (Greeks)
        full_grads = torch.autograd.grad(
            outputs=pred_price,
            inputs=x,
            grad_outputs=torch.ones_like(pred_price),
            create_graph=True, # OBS detta är inte för att få greeks utan för att beräkna gradienten av greeksen för optimeraren!!
            # retain_graph=True, är sant automatiskt
            only_inputs=True
        )[0]

        # Greek loss
        pred_grads = full_grads[:, index_in_input]
        y_grads = y[:, index_in_output]
        mse_per_greek = torch.mean((pred_grads - y_grads)**2, dim=0)
        l_g = torch.sum(mse_per_greek * greek_weights_tensor)
            

        # Get greek weights
        lambda_p = weights['lambda_p']
        lambda_g = weights['lambda_g']
        
        # Loss function
        loss = (lambda_p * l_p) + (lambda_g * l_g)

        # Compute loss gradients and update parameters
        loss.backward()
        optimizer.step()

        return {"loss": loss.item(), "l_p": l_p.item(), "l_g": l_g.item()}

In [7]:
def val_step(model, criterion, batch: Tuple, weights):

    x, y = batch

    x.requires_grad_(True)

    # Price loss
    pred_price = model(x)
    y_price = y[:, 0:1]
    l_p = criterion(pred_price, y_price)
    
    # Compute gradients (Greeks)    
    full_grads = torch.autograd.grad(
            outputs=pred_price,
            inputs=x,
            grad_outputs=torch.ones_like(pred_price),
            create_graph=False, # Behövs ej vid test/validering (se komentar för training_step)
            retain_graph=False
        )[0]
        
    # Greek loss
    pred_grads = full_grads[:, index_in_input]
    y_grads = y[:, index_in_output]
    mse_per_greek = torch.mean((pred_grads - y_grads)**2, dim=0) 
    l_g = torch.sum(mse_per_greek * greek_weights_tensor)


    # Get loss weights
    lambda_p = weights['lambda_p']
    lambda_g = weights['lambda_g']

    # Loss function
    loss = (lambda_p * l_p) + (lambda_g * l_g)

    return {"loss": loss.item(), "l_p": l_p.item(), "l_g": l_g.item()}

**HYPERPARAMETERS**

In [8]:
# Loss weights
lambda_p = 10
lambda_g = 1

# Model parameters 
nr_input_dim = 8
nr_hidden_layers = 7
nr_neurons = 250

# Optimizer settings
learning_rate = 3e-4
weight_decay = 1e-5
scheduler_patience = 7
scheduler_factor = 0.5

# Training loop
EPOCHS = 450
patience_early_stopping = 450

In [9]:
# initialisera modellen till device
model = DDN(input_dim=nr_input_dim, hidden_layers=nr_hidden_layers, neurons=nr_neurons).to(device)

# Loss + Optimizer
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
# Överväg att sänka lr och ska vi ha L-BFGS???
#scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=scheduler_patience, factor=scheduler_factor) # bestämer hur och när vi ska ändra lr 
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=150, T_mult=1, eta_min=1e-6)

**TRAINING LOOP**

In [10]:

# Path for saving best model state
import heston_project
PKG_DIR = Path(heston_project.__file__).resolve().parent
SAVE_DIR = PKG_DIR / "models" / "saved"
out_path = SAVE_DIR / "DDN_P_curriculum.pth"

# Tracker for early stopping
counter = 0

# Loss dicts
train_losses = {'total_train_loss': [], 'lp_train': [], 'lg_train': []}
val_losses = {'total_val_loss': [], 'lp_val': [], 'lg_val': []}

best_val_loss = 1000

#Loss weights
weights_dict = {'lambda_p': lambda_p, 'lambda_g': lambda_g}

print(f'Starting Training on {device}...')
start_time = time.time()        ##########


for epoch in range(EPOCHS):

    if epoch == 50:
        weights_dict['lambda_g'] = 1
        weights_dict['lambda_p'] = 1
    elif epoch == 100:
        weights_dict['lambda_g'] = 10
        weights_dict['lambda_p'] = 1
    
    model.train()

    running_tot_loss, running_lp_loss, running_lg_loss = 0.0, 0.0, 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        losses = train_step(model, criterion, batch = (batch_X, batch_y), optimizer = optimizer, weights = weights_dict)

        running_tot_loss += losses['loss']
        running_lp_loss += losses['l_p']
        running_lg_loss += losses['l_g']
    
    avg_train_total_loss = running_tot_loss/len(train_loader)
    avg_train_lp_loss = running_lp_loss/len(train_loader)
    avg_train_lg_loss = running_lg_loss/len(train_loader)

    train_losses['total_train_loss'].append(avg_train_total_loss)
    train_losses['lp_train'].append(avg_train_lp_loss)
    train_losses['lg_train'].append(avg_train_lg_loss)

    model.eval()
    running_tot_loss, running_lp_loss, running_lg_loss = 0.0, 0.0, 0.0


    # OBS ANVÄND ABSOLUT INTE TORCH NO GRAD
    for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            losses = val_step(model, criterion = criterion, batch = (batch_X, batch_y), weights = weights_dict)

            running_tot_loss += losses['loss']
            running_lp_loss += losses['l_p']
            running_lg_loss += losses['l_g']
    
    avg_val_total_loss = running_tot_loss/len(val_loader)
    avg_val_lp_loss = running_lp_loss/len(val_loader)
    avg_val_lg_loss = running_lg_loss/len(val_loader)


    val_losses['total_val_loss'].append(avg_val_total_loss)
    val_losses['lp_val'].append(avg_val_lp_loss)
    val_losses['lg_val'].append(avg_val_lg_loss)

    # Uppdatera Learning Rate
    scheduler.step()

    if avg_val_total_loss < best_val_loss:
        best_val_loss = avg_val_total_loss
        best_loss_price = avg_val_lp_loss
        best_loss_greeks = avg_val_lg_loss
        torch.save(model.state_dict(), out_path) # Save the best version
        counter = 0 # Återställ!!!
    else:
        counter += 1
        if counter >= patience_early_stopping:
            print(f"Early stopping triggered at epoch {epoch}")
            break
    
    if (epoch + 1) % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | LR: {current_lr:.2e}")
        print(f"  Train -> Tot: {avg_train_total_loss:.4e} | Price: {avg_train_lp_loss:.4e} | Greeks: {avg_train_lg_loss:.4e}")
        print(f"  Val   -> Tot: {avg_val_total_loss:.4e} | Price: {avg_val_lp_loss:.4e} | Greeks: {avg_val_lg_loss:.4e}")
        print("-" * 70)

print("*" * 70)
print(f"Best Val Loss -> Tot: {best_val_loss:.4e} | Price: {best_loss_price:.4e} | Greeks: {best_loss_greeks:.4e}")
print("*" * 70)

loss_df = pd.DataFrame({
    "epoch": range(1, len(train_losses["total_train_loss"]) + 1),
    "train_total_loss": train_losses["total_train_loss"],
    "train_lp_loss": train_losses["lp_train"],
    "train_lg_loss": train_losses["lg_train"],
    "val_total_loss": val_losses["total_val_loss"],
    "val_lp_loss": val_losses["lp_val"],
    "val_lg_loss": val_losses["lg_val"],
})

loss_df.to_csv("losses/loss_data/DDN_P_loss_history_curriculum.csv", index=False)

# Old results 10x price loss:   pval 3.2426871410251578e-09, gval 6.476634553109762e-08
# Old res old weights:          pval 2.8672840102217377e-09, gval 6.614701071328e-08
# OLd:                          Price: 1.5822e-09 | Greeks: 2.6473e-08

Starting Training on cpu...
Epoch [  5/450] | LR: 2.99e-04
  Train -> Tot: 2.9608e-04 | Price: 2.3511e-05 | Greeks: 6.0970e-05
  Val   -> Tot: 1.5956e-04 | Price: 1.1568e-05 | Greeks: 4.3882e-05
----------------------------------------------------------------------
Epoch [ 10/450] | LR: 2.97e-04
  Train -> Tot: 1.1330e-04 | Price: 9.0603e-06 | Greeks: 2.2701e-05
  Val   -> Tot: 5.8162e-05 | Price: 4.2068e-06 | Greeks: 1.6094e-05
----------------------------------------------------------------------
Epoch [ 15/450] | LR: 2.93e-04
  Train -> Tot: 8.0430e-05 | Price: 6.7968e-06 | Greeks: 1.2462e-05
  Val   -> Tot: 2.1217e-05 | Price: 1.3482e-06 | Greeks: 7.7345e-06
----------------------------------------------------------------------
Epoch [ 20/450] | LR: 2.87e-04
  Train -> Tot: 4.0746e-05 | Price: 3.3907e-06 | Greeks: 6.8395e-06
  Val   -> Tot: 2.0491e-05 | Price: 1.4464e-06 | Greeks: 6.0273e-06
----------------------------------------------------------------------
Epoch [ 25/450] | LR

In [ ]:
loss_df = pd.DataFrame({
    "epoch": range(1, len(train_losses["total_train_loss"]) + 1),
    "train_total_loss": train_losses["total_train_loss"],
    "train_lp_loss": train_losses["lp_train"],
    "train_lg_loss": train_losses["lg_train"],
    "val_total_loss": val_losses["total_val_loss"],
    "val_lp_loss": val_losses["lp_val"],
    "val_lg_loss": val_losses["lg_val"],
})

loss_df.to_csv("losses/loss_data/DDN_P_loss_history.csv", index=False)
